In [327]:
import pandas as pd

df_train = pd.read_csv("train.csv")
df_test = pd.read_csv("test.csv")

df_train.head()

,id,an,materie,liceu,judet,medie,procent_reusita,numar_candidati,preferinta_materie,anomalie
0,1,2014,"ANATOMIE SI FIZIOLOGIE UMANA, GENETICA SI ECOL...","LICEUL TEHNOLOGIC AGRICOL ""MIHAIL KOGALNICEANU...",Iași,10.00,100.0,1,2.9,0
1,2,2014,"ANATOMIE SI FIZIOLOGIE UMANA, GENETICA SI ECOL...",LICEUL TEORETIC GATAIA,Timiș,10.00,100.0,1,1.3,0
2,3,2014,"ANATOMIE SI FIZIOLOGIE UMANA, GENETICA SI ECOL...","COLEGIUL NATIONAL ""GHEORGHE SINCAI"" CLUJ-NAPOCA",Cluj,9.85,100.0,2,1.7,0
3,4,2014,"ANATOMIE SI FIZIOLOGIE UMANA, GENETICA SI ECOL...","LICEUL TEORETIC ""ORBAN BALAZS"" CRISTURU SECUIESC",Harghita,9.80,100.0,1,1.8,1
4,5,2014,"ANATOMIE SI FIZIOLOGIE UMANA, GENETICA SI ECOL...","COLEGIUL NATIONAL ""GHEORGHE MUNTEANU MURGOCI""",Brăila,9.77,100.0,6,3.4,0


In [328]:
og_X = df_train[['an', 'materie', 'liceu', 'judet', 'procent_reusita', 'numar_candidati', 'preferinta_materie']]
og_X_final = df_test[['an', 'materie', 'liceu', 'judet', 'procent_reusita', 'numar_candidati', 'preferinta_materie']]

In [329]:
max_pref = df_test[df_test.liceu == 'COLEGIUL NATIONAL "UNIREA" FOCSANI'].preferinta_materie.max()
subtask1 = df_test[(df_test.liceu == 'COLEGIUL NATIONAL "UNIREA" FOCSANI') & (df_test.preferinta_materie >= 50)].materie

df_1 = pd.DataFrame({
    "id": 1,
    "subtaskID": [1],
    "answer": subtask1.iloc[2]
})

subtask1

3565                  GENERAL
7860             LIMBA ROMANA
10122    MATEMATICA MATE-INFO
Name: materie, dtype: object

In [330]:
date_focsani = df_train[(df_train.liceu == 'COLEGIUL NATIONAL "UNIREA" FOCSANI') & (df_train.materie == "INFORMATICA MI C-C++")]
pop_pe_an = date_focsani.groupby('an').preferinta_materie.mean()
subtask2 = pop_pe_an.idxmax()

df_2 = pd.DataFrame({
    "id": 2,
    "subtaskID": [2],
    "answer": subtask2
})

df_2

,id,subtaskID,answer
0,2,2,2022


In [331]:
df_train.columns

Index(['id', 'an', 'materie', 'liceu', 'judet', 'medie', 'procent_reusita',
       'numar_candidati', 'preferinta_materie', 'anomalie'],
      dtype='object')

In [332]:
from sklearn.preprocessing import TargetEncoder #target encoding pt ca sunt foarte multe valori ofc

to_enc = ['materie', 'liceu', 'judet']

for col in to_enc:
    enc = TargetEncoder()
    enc.fit(df_train[[col]], df_train["medie"])
    df_train[col] = enc.transform(df_train[[col]])
    df_test[col] = enc.transform(df_test[[col]])

df_train.head()

,id,an,materie,liceu,judet,medie,procent_reusita,numar_candidati,preferinta_materie,anomalie
0,1,2014,7.11492,5.965442,7.167490,10.00,100.0,1,2.9,0
1,2,2014,7.11492,6.553024,6.638916,10.00,100.0,1,1.3,0
2,3,2014,7.11492,9.066372,7.475090,9.85,100.0,2,1.7,0
3,4,2014,7.11492,7.359272,6.496818,9.80,100.0,1,1.8,1
4,5,2014,7.11492,9.038640,6.791678,9.77,100.0,6,3.4,0


In [333]:
features = ['an', 'materie', 'liceu', 'judet', 'procent_reusita', 'numar_candidati', 'preferinta_materie']
target1 = 'medie'
target2 = 'anomalie'

X = df_train[features]
y1 = df_train[target1]
y2 = df_train[target2]

X_final = df_test[features]

from sklearn.preprocessing import RobustScaler

scaler = RobustScaler()

#X = pd.DataFrame(scaler.fit_transform(X), columns=features)
#X_final = pd.DataFrame(scaler.transform(X_final), columns=features)

X.head()

,an,materie,liceu,judet,procent_reusita,numar_candidati,preferinta_materie
0,2014,7.11492,5.965442,7.167490,100.0,1,2.9
1,2014,7.11492,6.553024,6.638916,100.0,1,1.3
2,2014,7.11492,9.066372,7.475090,100.0,2,1.7
3,2014,7.11492,7.359272,6.496818,100.0,1,1.8
4,2014,7.11492,9.038640,6.791678,100.0,6,3.4


In [334]:
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.svm import LinearSVC

model = XGBClassifier()

cross_val_score(model, X, y2, cv=5, scoring='roc_auc').mean()

0.9169962618273513

In [335]:
model.fit(X, y2)
y2_preds = model.predict_proba(X_final)[:, 1]

In [336]:
df_4 = pd.DataFrame({
    "id": df_test.id,
    "subtaskID": 4,
    "answer": y2_preds
})

df_4.head()

,id,subtaskID,answer
0,120760,4,0.189830
1,120761,4,0.262831
2,120762,4,0.000432
3,120763,4,0.059797
4,120764,4,0.251676


In [339]:
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
from sklearn.model_selection import cross_val_score

model = CatBoostRegressor(silent=True, cat_features=to_enc)

-cross_val_score(model, og_X, y1, cv=3, scoring='neg_mean_absolute_error').mean()

0.5004923969939331

In [342]:
model.fit(og_X, y1)
y1_preds = model.predict(og_X_final)

In [343]:
df_3 = pd.DataFrame({
    "id": df_test.id,
    "subtaskID": 3,
    "answer": y1_preds
})

df_3.head()

,id,subtaskID,answer
0,120760,3,8.010153
1,120761,3,6.959733
2,120762,3,8.513444
3,120763,3,8.967374
4,120764,3,8.346055


In [344]:
submission = pd.concat([df_1, df_2, df_3, df_4])

submission

,id,subtaskID,answer
0,1,1,MATEMATICA MATE-INFO
0,2,2,2022
0,120760,3,8.010153
1,120761,3,6.959733
2,120762,3,8.513444
...,...,...,...
13012,133772,4,0.802332
13013,133773,4,0.92813
13014,133774,4,0.533744
13015,133775,4,0.970317


In [345]:
submission.to_csv("submission.csv", index=False)